# Modelagem - Credit Card Fraud Detection Dataset


# Preparação inicial dos dados para modelagem

## 1. Importando o dataset via Kaggle

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'creditcardfraud' dataset.
Path to dataset files: /kaggle/input/creditcardfraud


## 2. Garantindo integridade dos dados.
- Tranformar 'Time' para 'Hour'
- Normalizar 'Amount'
- Remover duplicatas

In [3]:
from sklearn.preprocessing import RobustScaler
import pandas as pd
import os

# Integridade e Carregamento
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
full_path = os.path.join(path, csv_file)
df = pd.read_csv(full_path)

# Remoção de duplicatas
duplicados = df.duplicated().sum()
print(f"\n--- Duplicatas: {duplicados} ---")
df = df.drop_duplicates() if duplicados > 0 else df
print(f"Formato após limpeza: {df.shape}")

# Converter segundos para horas (0-23)
df['Hour'] = (df['Time'] // 3600) % 24

# Garantir que a coluna Hour existe
if 'Hour' not in df.columns:
    df['Hour'] = (df['Time'] // 3600) % 24

# Normalização do 'Amount'
# Verificamos se a coluna Amount existe para evitar erro em re-execuções
if 'Amount' in df.columns:
    rs = RobustScaler()
    df['scaled_amount'] = rs.fit_transform(df['Amount'].values.reshape(-1,1))

    # Remover as colunas originais que não usaremos mais
    # Verificamos Time também por segurança
    cols_to_drop = [c for c in ['Time', 'Amount'] if c in df.columns]
    df.drop(cols_to_drop, axis=1, inplace=True)

    # Reordenar para deixar o scaled_amount no início
    scaled_amount = df['scaled_amount']
    df.drop(['scaled_amount'], axis=1, inplace=True)
    df.insert(0, 'scaled_amount', scaled_amount)

print("Colunas atuais no DataFrame:", df.columns.tolist())
display(df.head())


--- Duplicatas: 1081 ---
Formato após limpeza: (283726, 31)
Colunas atuais no DataFrame: ['scaled_amount', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Class', 'Hour']


,scaled_amount,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Class,Hour
0,1.774718,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,0,0.0
1,-0.268530,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,0,0.0
2,4.959811,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,0,0.0
3,1.411487,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,0,0.0
4,0.667362,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,0,0.0


# Próximos Passos e Estratégia de Modelagem

Para as próximas etapas, seguiremos o seguinte roteiro para garantir a máxima performance na detecção de fraudes:

1.  **Divisão Estratificada**: Realizar o split entre treino e teste (80/20) garantindo que a proporção de fraudes seja mantida em ambos os sets.
2.  **Cost-Sensitive Learning**: Em vez de balancear o dataset (o que pode remover informações valiosas das transações legítimas), utilizaremos o parâmetro `scale_pos_weight` no **XGBoost**. Isso forçará o modelo a penalizar severamente erros cometidos na classe minoritária (Fraude).
3.  **Otimização com Optuna**: Utilizar busca de hiperparâmetros focada exclusivamente na métrica **PR-AUC** (Area Under the Precision-Recall Curve), que é mais robusta para datasets desbalanceados do que a acurácia ou a curva ROC.
4.  **Calibração de Probabilidades**: Implementar **Platt Scaling** ou **Isotonic Regression** para calibrar as saídas do modelo, garantindo que as probabilidades estimadas reflitam o risco real para a operação de negócio.

## 1. Benchmarking: Comparação de Modelos

Para escolher o modelo que melhor se adeque ao nosso dataset e ao problema que queremos resolver, comparamos a performance de três diferentes modelos:

1. **Regressão Logística (Baseline):** Um modelo linear simples usado como ponto de partida para medir a complexidade do problema.
2. **Random Forest (Ensemble):** Um conjunto de árvores de decisão que utiliza `class_weight='balanced'` para tentar compensar a raridade das fraudes.
3. **XGBoost (Gradient Boosting):** Modelo de alto desempenho que utiliza **Cost-Sensitive Learning** através do parâmetro `scale_pos_weight`. Diferente de outros métodos, ele penaliza erros na classe minoritária proporcionalmente ao desbalanceamento, permitindo alta revocação sem a necessidade de reamostragem (oversampling/undersampling).

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, average_precision_score

# 1. Preparação dos dados e Divisão Estratificada
X = df.drop('Class', axis=1)
y = df['Class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Cálculo do scale_pos_weight para o XGBoost
count_legit = (y_train == 0).sum()
count_fraud = (y_train == 1).sum()
spw = count_legit / count_fraud

# --- MODELO 1: Regressão Logística ---
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)
y_pred_log = log_reg.predict(X_test)
y_probs_log = log_reg.predict_proba(X_test)[:, 1]
auccpr_log = average_precision_score(y_test, y_probs_log)

# --- MODELO 2: Random Forest ---
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
y_probs_rf = rf_model.predict_proba(X_test)[:, 1]
auccpr_rf = average_precision_score(y_test, y_probs_rf)

# --- MODELO 3: XGBoost (Cost-Sensitive) ---
xgb_model = XGBClassifier(
    scale_pos_weight=spw,
    learning_rate=0.1,
    n_estimators=100,
    max_depth=6,
    random_state=42,
    eval_metric='aucpr'
)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
y_probs_xgb = xgb_model.predict_proba(X_test)[:, 1]
auccpr_xgb = average_precision_score(y_test, y_probs_xgb)

# --- RESUMO COMPARATIVO ---
print("\n--- Relatório: Regressão Logística ---")
print(classification_report(y_test, y_pred_log))

print("\n--- Relatório: Random Forest ---")
print(classification_report(y_test, y_pred_rf))

print("\n--- Relatório: XGBoost ---")
print(classification_report(y_test, y_pred_xgb))

print("=== PERFORMANCE COMPARATIVA (PR-AUC) ===")
print(f"Regressão Logística: {auccpr_log:.4f}")
print(f"Random Forest:       {auccpr_rf:.4f}")
print(f"XGBoost:             {auccpr_xgb:.4f} (Vencedor)")


--- Relatório: Regressão Logística ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.85      0.59      0.70        95

    accuracy                           1.00     56746
   macro avg       0.92      0.79      0.85     56746
weighted avg       1.00      1.00      1.00     56746


--- Relatório: Random Forest ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.97      0.69      0.81        95

    accuracy                           1.00     56746
   macro avg       0.99      0.85      0.90     56746
weighted avg       1.00      1.00      1.00     56746


--- Relatório: XGBoost ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.83      0.79      0.81        95

    accuracy                           1.00     56746
   macro avg       0.92   

### 1.1. Justificativa da Escolha do Modelo (XGBoost)

A escolha do **XGBoost** (PR-AUC: 0.8103) como modelo final, superando a Regressão Logística (0.6936) e o Random Forest (0.8047), baseia-se nos seguintes pontos:

*   **Superioridade em PR-AUC:** Em problemas de detecção de fraude, a métrica **PR-AUC (Area Under the Precision-Recall Curve)** é muito mais relevante que a Acurácia ou o F1-Score isolado. Como o dataset é extremamente desbalanceado (99.8% legítimas), a acurácia é enganosa. O PR-AUC foca especificamente na performance da classe minoritária (Fraude), avaliando o quão bem o modelo consegue separar as classes em diferentes limiares de decisão.
*   **Equilíbrio entre Precisão e Revocação (Recall):** Embora o Random Forest tenha apresentado uma Precisão ligeiramente maior (0.97 vs 0.83), o XGBoost entregou uma **Revocação significativamente melhor (0.79 vs 0.69)**. Para o negócio, é preferível investigar alguns falsos positivos extras do que deixar passar 10% a mais de fraudes reais.
*   **Por que não F1-Score?** O F1-Score assume um peso igual entre Precisão e Recall. No entanto, o PR-AUC nos dá uma visão holística da capacidade de ranking do modelo, sendo mais estável para otimização quando lidamos com proporções tão pequenas de eventos positivos.
*   **Robustez ao Desbalanceamento via Cost-Sensitive Learning:** Através do parâmetro `scale_pos_weight`, o XGBoost foi capaz de aprender as características complexas das fraudes sem a necessidade de gerar dados artificiais (SMOTE), capturando as não-linearidades que identificamos previamente na visualização t-SNE.

# 3. Otimização de Hiperparâmetros (Optuna)

Busca Bayesiana pelos hiperparâmetros que maximizam o PR-AUC, usando o conjunto de treino **sem duplicatas** definido no benchmark (seção 1).

> **Protocolo sem vazamento + validação cruzada:** cada conjunto de hiperparâmetros é avaliado por **validação cruzada estratificada** (`StratifiedKFold`) *dentro do conjunto de treino*. Isso (a) mantém o **teste intocado** durante toda a busca e (b) evita o overfitting a um único conjunto de validação pequeno — o score de cada *trial* é a **média do PR-AUC entre os folds**, uma estimativa mais estável. O teste é usado uma só vez, na avaliação final (seção 3.1).

A otimização focará em:
1.  **Complexidade da Árvore**: `max_depth` e `min_child_weight`.
2.  **Robustez**: `subsample` e `colsample_bytree` para evitar overfitting.
3.  **Regularização**: `gamma`, `alpha` (L1) e `lambda` (L2).

In [5]:
!pip install optuna -q

import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
import numpy as np

# Validação cruzada estratificada (dados já sem duplicatas -> não é preciso agrupar)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

def objective(trial):
    # Espaço de busca de hiperparâmetros (scale_pos_weight é definido por fold, abaixo)
    param = {
        'verbosity': 0,
        'objective': 'binary:logistic',
        'eval_metric': 'aucpr',
        'random_state': 42,
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
    }

    fold_scores = []
    # CV sobre o TREINO (X_train/y_train do benchmark) — o teste nunca é tocado
    for tr_idx, va_idx in cv.split(X_train, y_train):
        X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]

        # peso recalculado DENTRO de cada fold (sem olhar a parte de validação)
        spw_fold = (y_tr == 0).sum() / (y_tr == 1).sum()
        model = XGBClassifier(**param, scale_pos_weight=spw_fold)
        model.fit(X_tr, y_tr)

        preds = model.predict_proba(X_va)[:, 1]
        fold_scores.append(average_precision_score(y_va, preds))

    # Objetivo = PR-AUC médio entre os folds (estimativa estável, menos sujeita a ruído)
    return np.mean(fold_scores)

# Criar e executar o estudo (CV deixa cada trial ~3x mais lento; timeout ampliado)
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, timeout=1800)

print("\n--- Otimização Concluída (CV 3-fold) ---")
print(f"Melhor PR-AUC médio na CV: {study.best_value:.4f}")
print("Melhores parâmetros:", study.best_params)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.1 MB/s eta 0:00:00


[I 2026-06-24 22:11:18,945] A new study created in memory with name: no-name-d178d2a8-1ced-4be2-9775-433e88e53792
[I 2026-06-24 22:11:29,310] Trial 0 finished with value: 0.8479944224052495 and parameters: {'n_estimators': 144, 'max_depth': 7, 'learning_rate': 0.11201599199344592, 'subsample': 0.6862002846658577, 'colsample_bytree': 0.6970058467784491, 'min_child_weight': 5, 'gamma': 0.16751775325323578, 'alpha': 6.420860567410615e-07, 'lambda': 2.9061398710598636e-08}. Best is trial 0 with value: 0.8479944224052495.
[I 2026-06-24 22:11:36,698] Trial 1 finished with value: 0.8260828820155849 and parameters: {'n_estimators': 146, 'max_depth': 3, 'learning_rate': 0.1917867682666677, 'subsample': 0.5873342166966886, 'colsample_bytree': 0.7895670910816767, 'min_child_weight': 1, 'gamma': 4.640247014117974e-05, 'alpha': 9.624262355879837e-07, 'lambda': 0.009421891868567591}. Best is trial 0 with value: 0.8479944224052495.
[I 2026-06-24 22:11:45,433] Trial 2 finished with value: 0.8108302935


--- Otimização Concluída (CV 3-fold) ---
Melhor PR-AUC médio na CV: 0.8491
Melhores parâmetros: {'n_estimators': 186, 'max_depth': 5, 'learning_rate': 0.11132582568590109, 'subsample': 0.8134183469660429, 'colsample_bytree': 0.8500312794859216, 'min_child_weight': 5, 'gamma': 5.200168262602653e-07, 'alpha': 0.00014299522135105303, 'lambda': 3.6000018332442216e-07}


### 3.1. Treinamento e avaliação do modelo otimizado

Aplicamos os melhores hiperparâmetros encontrados pela busca para treinar o modelo otimizado no **treino completo** e avaliá-lo, **uma única vez**, no conjunto de teste. O resultado é comparado ao **XGBoost do benchmark** (sem otimização) — a decisão sobre qual usar como detector final é discutida na seção 3.3.

In [6]:
# Modelo final com os melhores hiperparâmetros (selecionados por validação cruzada).
# Reajuste no treino COMPLETO para aproveitar todos os dados de treino.
best_params = dict(study.best_params)   # cópia, para não mutar o estudo
best_params['scale_pos_weight'] = spw   # peso do treino (benchmark, sem duplicatas)
best_params['eval_metric'] = 'aucpr'

final_xgb = XGBClassifier(**best_params, random_state=42)
final_xgb.fit(X_train, y_train)

# Avaliação Final: o conjunto de TESTE é usado aqui pela primeira e única vez (sem vazamento)
y_probs_final = final_xgb.predict_proba(X_test)[:, 1]
auccpr_final = average_precision_score(y_test, y_probs_final)

print(f"PR-AUC médio na CV (seleção dos hiperparâmetros): {study.best_value:.4f}")
print(f"PR-AUC do XGBoost benchmark (sem otimização):     {auccpr_xgb:.4f}")
print(f"PR-AUC Final no TESTE (modelo otimizado):         {auccpr_final:.4f}")
print(f"Ganho sobre o benchmark:                          {auccpr_final - auccpr_xgb:.4f}")

PR-AUC médio na CV (seleção dos hiperparâmetros): 0.8491
PR-AUC do XGBoost benchmark (sem otimização):     0.8103
PR-AUC Final no TESTE (modelo otimizado):         0.8099
Ganho sobre o benchmark:                          -0.0004


### 3.2. Teste de robustez: o ganho é real ou ruído?

O resultado de um único split é frágil: com ~95 fraudes no teste, diferenças de PR-AUC da ordem de ±0.01 podem ser puro acaso da divisão. Para checar, reavaliamos o modelo **otimizado** contra **dois baselines**, em **5 divisões diferentes** (variando a *seed*, com split estratificado):

- **XGBoost default** — parâmetros padrão da biblioteca + `scale_pos_weight`;
- **XGBoost benchmark** — a configuração manual da seção 1 (`lr=0.1, n_est=100, max_depth=6`).

Comparar com os dois é importante porque *"ganhar do benchmark"* e *"ganhar de um bom default"* são perguntas diferentes. Regra de leitura: se o **ganho médio for menor que o desvio-padrão**, ele é **indistinguível de zero** (ruído).

In [7]:
# Teste de robustez: o ganho do modelo otimizado é real ou ruído?
# Comparamos o modelo OTIMIZADO contra DOIS baselines, em 5 divisões treino/teste
# (split estratificado, variando a seed):
#   - DEFAULT:   XGBoost com parâmetros padrão + scale_pos_weight
#   - BENCHMARK: a config manual da seção 1 (lr=0.1, n_est=100, max_depth=6)
import numpy as np
from sklearn.model_selection import train_test_split

# hiperparâmetros otimizados, removendo chaves que passamos explicitamente abaixo
opt_params = {k: v for k, v in study.best_params.items()
              if k not in ('scale_pos_weight', 'eval_metric', 'random_state')}

g_default, g_bench = [], []
for seed in range(5):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    spw_r = (ytr == 0).sum() / (ytr == 1).sum()

    def pr_auc(model):
        model.fit(Xtr, ytr)
        return average_precision_score(yte, model.predict_proba(Xte)[:, 1])

    ap_default = pr_auc(XGBClassifier(scale_pos_weight=spw_r, eval_metric='aucpr', random_state=42))
    ap_bench   = pr_auc(XGBClassifier(scale_pos_weight=spw_r, learning_rate=0.1, n_estimators=100,
                                      max_depth=6, eval_metric='aucpr', random_state=42))
    ap_opt     = pr_auc(XGBClassifier(**opt_params, scale_pos_weight=spw_r,
                                      eval_metric='aucpr', random_state=42))

    g_default.append(ap_opt - ap_default)
    g_bench.append(ap_opt - ap_bench)
    print(f"seed {seed}: otim={ap_opt:.4f} | otim-default={ap_opt-ap_default:+.4f} | otim-benchmark={ap_opt-ap_bench:+.4f}")

def veredito(diffs, nome):
    m, s = np.mean(diffs), np.std(diffs)
    tag = "DENTRO DO RUÍDO" if abs(m) < s else ("GANHO consistente" if m > 0 else "PIORA consistente")
    print(f"Ganho otim vs {nome}: {m:+.4f} ± {s:.4f}  ->  {tag}")

print()
veredito(g_default, "DEFAULT  ")
veredito(g_bench,   "BENCHMARK")

seed 0: otim=0.8716 | otim-default=-0.0038 | otim-benchmark=+0.0065
seed 1: otim=0.7889 | otim-default=-0.0085 | otim-benchmark=+0.0090
seed 2: otim=0.8456 | otim-default=+0.0130 | otim-benchmark=+0.0210
seed 3: otim=0.8008 | otim-default=+0.0076 | otim-benchmark=+0.0309
seed 4: otim=0.8168 | otim-default=-0.0120 | otim-benchmark=+0.0009

Ganho otim vs DEFAULT  : -0.0007 ± 0.0095  ->  DENTRO DO RUÍDO
Ganho otim vs BENCHMARK: +0.0137 ± 0.0108  ->  GANHO consistente


### 3.3. Análise Crítica: Vale a pena usar Optuna?

O teste de robustez (seção 3.2) revela um resultado sutil e importante, comparando o modelo otimizado contra dois baselines:

- **vs. XGBoost default:** o ganho médio é **praticamente zero** (dentro do ruído). O modelo otimizado **não supera** um XGBoost com parâmetros padrão.
- **vs. XGBoost benchmark (`lr=0.1`):** há um **ganho pequeno e razoavelmente consistente** (~+0.01 de PR-AUC).

Note ainda que, no **split único** (seção 3.1), o otimizado e o benchmark praticamente empataram, enquanto na **média de 5 seeds** o otimizado supera o benchmark por ~+0.01 — um lembrete concreto de que um único split pode enganar.

#### Como interpretar essa diferença
A chave é que a configuração do **benchmark (seção 1) usava `learning_rate=0.1`, um valor conservador** para este dataset — o próprio default da biblioteca (`lr=0.3`) já performa melhor. O Optuna encontrou parâmetros que **igualam um bom default**, mas **não vão além** dele. Ou seja: o "ganho sobre o benchmark" não é mérito do tuning sofisticado — é apenas o tuning corrigindo uma escolha manual subótima de hiperparâmetro.

#### A lição mais importante (por que o teste de robustez foi decisivo)
Um único split — ou um único baseline — engana. Só comparando contra um **baseline forte** e repetindo em **várias seeds** dá para ver que a diferença real é indistinguível de zero. Nunca se deve concluir a partir de uma divisão única nem de um baseline fraco quando a classe positiva é rara e a variância das métricas é alta.

#### Sobre a estimativa de validação
O PR-AUC da validação cruzada (~0.85) é um pouco mais alto que o do teste em um split específico (~0.81), mas fica **dentro da faixa de valores observada no teste entre as diferentes seeds** (~0.79–0.87). Ou seja, a CV não superestima de forma exagerada — bem diferente da versão inicial (com validação única), em que a validação inflava muito o resultado. A variância entre seeds é grande justamente porque há poucas fraudes no teste (~95).

#### Lições de método
- **PR-AUC** é a métrica certa para dados tão desbalanceados; a acurácia seria enganosa.
- **Cost-sensitive learning** (`scale_pos_weight`) trata o desbalanceamento sem gerar dados sintéticos (SMOTE) nem descartar transações.
- O impacto de negócio vem mais do **threshold e da calibração** (próximas seções) do que do ajuste fino de hiperparâmetros.

**Conclusão:** para este problema, a otimização de hiperparâmetros **não entrega ganho real** além do que um XGBoost bem configurado já oferece. Adotamos o **modelo mais simples** como detector final e mantemos o Optuna como **experimento documentado** — cuja maior contribuição foi metodológica: medir corretamente (sem vazamento, com baseline forte e múltiplas seeds), para não confundir a correção de um hiperparâmetro subótimo com um ganho real.